# unit01 レッスン: HTMLの構造とPythonの足場

**このレッスンで作れるようになるもの**: 生のHTML文字列から「見出しの文字」「リンクのURL」「タグ一覧」を素朴な文字列処理で拾い出し、リストと辞書に整形する — スクレイピングの最小パイプラインの手触り。

これは以降の全ユニットの土台です。unit02 で使う BeautifulSoup も、内部でやっているのは結局この「文字列を切り出す処理」です。まず素の力で一度やっておくと、ライブラリが何を肩代わりしてくれているのかが腹落ちします。

- 所要時間: 15〜25分
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok

# このレッスンで題材にするHTML(演習の data/simple_page.html と同じ内容)。
# 本物のスクレイピングでは requests でネットから取ってきた文字列がこの変数に入る、
# と想像してください。今は「取得済みの文字列」として手元に用意しておきます。
SIMPLE_PAGE = """<!DOCTYPE html>
<html lang="ja">
<head>
    <meta charset="UTF-8">
    <title>喫茶ポラリス | お知らせ</title>
</head>
<body>
    <header>
        <h1>喫茶ポラリス</h1>
        <p>営業時間: 8:00-19:00 / 定休日: 水曜</p>
    </header>
    <main>
        <h2>お知らせ</h2>
        <p>10月から新しい季節限定ブレンドを提供します。豆は毎朝店内で焙煎しています。</p>
        <p>席数が12席から18席に増えました。予約は下記のリンクからどうぞ。</p>
        <ul>
            <li><a href="/menu">メニュー一覧</a></li>
            <li><a href="/reserve">予約フォーム</a></li>
            <li><a href="/access">アクセス</a></li>
        </ul>
    </main>
    <footer>
        <p>お問い合わせ: info@example-cafe.test</p>
    </footer>
</body>
</html>"""

# スタッフのプロフィールカード(演習の data/profile_card.html と同じ内容)
PROFILE_CARD = """<!DOCTYPE html>
<html lang="ja">
<head>
    <meta charset="UTF-8">
    <title>スタッフ紹介 | 喫茶ポラリス</title>
</head>
<body>
    <div class="profile-card">
        <h2 class="name">名前: 田中 美咲</h2>
        <p class="job">職業: 焙煎士</p>
        <p class="bio">コーヒー豆の焙煎歴8年。浅煎りの酸味を活かした配合が得意です。</p>
        <p class="tags">タグ: 焙煎, ハンドドリップ, 浅煎り</p>
    </div>
</body>
</html>"""

print("準備OK! SIMPLE_PAGE は", len(SIMPLE_PAGE), "文字、PROFILE_CARD は", len(PROFILE_CARD), "文字")

---
## 概念1: スクレイピングの全体像と、HTMLの「入れ子」構造

### なぜ学ぶか
スクレイピングの仕事は、Webページから欲しい情報(価格・商品名・リンク先)を機械的に集めることです。求人票の「データ収集の自動化」「Webからの情報抽出」はほぼこれ。まず最初に**全体の地図**を頭に入れておくと、各ユニットで「今どこをやっているか」が迷子になりません。

### 解説

スクレイピングは、いつも次の**4工程のパイプライン**です:

```
[1 取得]  →  [2 解析]      →  [3 整形]        →  [4 出力]
HTTPで     HTMLの文字列から   list や dict に      CSVなどに
HTMLを     欲しい部分を        きれいに詰め直す      書き出す
もらう      切り出す
```

**このユニット(unit01)でやるのは主に [2 解析] と [3 整形] の素振り**です。[1 取得] のHTTP通信は unit04、[4 出力] のCSVは unit05 で扱います。今は「取得済みのHTML文字列」がすでに手元にある前提で進めます。

#### HTMLは「入れ子になった箱」= ツリー構造

HTMLは、`<タグ>中身</タグ>` という**箱**が入れ子になったものです。C# で言えば、オブジェクトが別のオブジェクトを子として持つ**オブジェクトグラフ**そのもの:

```
html                          class Html {
 ├─ head                        Head Head;      // 子オブジェクト
 │   └─ title "喫茶ポラリス"      Body Body;
 └─ body                       }
     ├─ header                 class Body {
     │   ├─ h1 "喫茶ポラリス"      Header Header;  // さらに子
     │   └─ p  "営業時間..."       Main   Main;
     └─ main                   }
         ├─ h2 "お知らせ"
         ├─ p  "10月から..."
         └─ ul → li → a "/menu"
```

親の中に子、子の中に孫。この木(ツリー)を **DOM (Document Object Model)** と呼びます。unit02 以降ではこの木を「オブジェクトとして」たどりますが、**まず今日は、この木がただの文字列としてどう見えるか**を確認します。

In [ ]:
# GOAL: HTMLが「タグの入れ子」でできた、ただの1本の文字列であることを目で見る

# STEP 1: SIMPLE_PAGE はただの文字列。先頭200文字を覗いてみる
print(SIMPLE_PAGE[:200])
print("-" * 40)

# STEP 2: 特定のタグが文字列の何文字目にあるか調べる
#         str.find("探す文字列") は最初に見つかった位置(先頭からの文字数)を返す。
#         見つからなければ -1(C# の string.IndexOf と同じ)
h1_pos = SIMPLE_PAGE.find("<h1>")
print("<h1> が現れる位置:", h1_pos)

# STEP 3: 入れ子の深さ = タグの数。<li> は3個あるはず(メニュー/予約/アクセス)
print("<li> の個数:", SIMPLE_PAGE.count("<li>"))

### 予測してみよう

次のセルは `SIMPLE_PAGE.find("<p>")`(最初の `<p>` の位置)と `SIMPLE_PAGE.count("<p>")`(`<p>` の総数)を表示します。

**実行する前に予測**: `<p>` はこのページに何個あるでしょう?(ヒント: 営業時間・お知らせ2つ・お問い合わせ)。そして `find` が返すのは「最初の1個」の位置だけであることも確認してください。

In [ ]:
# 予測してから実行!
print("最初の <p> の位置:", SIMPLE_PAGE.find("<p>"))
print("<p> の総数     :", SIMPLE_PAGE.count("<p>"))

`find` は「最初の1個」の位置しか返さない、`count` は「総数」を返す — この違いが後で効いてきます。

### 書いてみる

**課題**: `SIMPLE_PAGE` の中に `<a href=` という文字列(リンクの開始部分)が**何個あるか**を数えて `result1` に入れてください(期待値: `3`。メニュー/予約/アクセスの3リンク)。

ヒント(概念レベル): 「総数を数えるメソッド」を使うだけ。1行で書けます。

In [ ]:
result1 = None
# ここに書く(result1 に代入する)


check("概念1: タグを数える", result1, 3,
      hint='"総数" を返すのは find ではなく count。SIMPLE_PAGE.count("...") の形')

---
## 概念2: 文字列操作 — strip / find+スライス / split で「切り出す」

### なぜ学ぶか
ネットから取ってきた文字列は、前後に改行や空白がベッタリ付いていたり、`"営業時間: 8:00-19:00"` のように**ラベルと値がくっついて**いたりします。実務では「この汚れた文字列から値だけ抜く」処理を延々と書きます。ここが解析([2])の心臓部です。

### 解説

Python の文字列は、C# の `string` とほぼ同じメソッドを持っています。今日使う4つ:

| Python | 何をする | C# の対応 |
|--------|----------|-----------|
| `s.strip()` | 前後の空白・改行を除去 | `s.Trim()` |
| `s.find(x)` | `x` が最初に現れる位置(なければ-1) | `s.IndexOf(x)` |
| `s[a:b]` | a文字目からb文字目の**手前**まで切り出す(スライス) | `s.Substring(a, b-a)` |
| `s.split(sep)` | `sep` で区切ってリストにする | `s.Split(sep)` |

**スライス `s[a:b]`** が一番の新顔です。`b` は「含まない」— NumPy の `arange` と同じで**終端は含まない**のがPython流。C# の `Substring(開始, 長さ)` と違って**開始位置と終了位置**を書く点に注意。

**タグの中身を切り出す定石**:
1. `<h1>` の位置を `find` で探す → その**直後**からが中身
2. `</h1>` の位置を `find` で探す → その**直前**までが中身
3. `s[中身の開始:中身の終了]` でスライス

`split` は `"a, b, c".split(",")` → `["a", " b", " c"]` のようにリスト化します(各要素に空白が残る点は後で `strip` で掃除)。

In [ ]:
# GOAL: <h1>タグの中身「喫茶ポラリス」を、find と スライスだけで抜き出す

# STEP 1: 開始タグ <h1> の位置を探し、その「直後」を中身の開始位置にする
start_tag = "<h1>"
start = SIMPLE_PAGE.find(start_tag) + len(start_tag)   # タグの長さ分だけ後ろへ
print("中身の開始位置:", start)

# STEP 2: 終了タグ </h1> の位置 = 中身の終了位置
end = SIMPLE_PAGE.find("</h1>")
print("中身の終了位置:", end)

# STEP 3: スライスで切り出す(end は「含まない」ので </h1> 自体は入らない)
title_text = SIMPLE_PAGE[start:end]
print("抜き出した中身:", repr(title_text))

# --- おまけ: strip と split の動きも見ておく ---
raw = "  焙煎, ハンドドリップ, 浅煎り  "
print("strip前:", repr(raw))
print("strip後:", repr(raw.strip()))          # 前後の空白が消える
print("splitで分割:", raw.strip().split(","))  # カンマで3つに分かれる(各要素に空白が残る)

### 予測してみよう

次のセルは `"営業時間: 8:00-19:00"` を `split(":", 1)` で分割します。第2引数の `1` は「**最大1回だけ**分割する」という意味(C# の `Split(':', 2)` の「最大2要素」に相当)。

**実行する前に予測**: 結果のリストは何個の要素になり、`[1]`(値の側)は何になるでしょう? 時刻の中にも `:` があることに注意して考えてください。

In [ ]:
# 予測してから実行!
line = "営業時間: 8:00-19:00"
parts = line.split(":", 1)   # 最初の : でだけ分割(maxsplit=1)
print("分割結果:", parts)
print("値の側 :", repr(parts[1]))
print("整えた値:", repr(parts[1].strip()))

`maxsplit=1` があるおかげで `8:00` の `:` では割られず、`" 8:00-19:00"` が丸ごと値になりました。これを付け忘れると時刻でバラバラに割れてバグになります。

### 書いてみる

**課題**: `PROFILE_CARD` から `<h2 class="name">` タグの中身(`"名前: 田中 美咲"`)を抜き出し、さらに `"名前: "` を取り除いて**名前だけ**を `result2` に入れてください(期待値: `"田中 美咲"`)。

ヒント(概念レベル): (1) 開始タグ `'<h2 class="name">'` と終了タグ `"</h2>"` を `find`+スライスで中身を切り出す → (2) その文字列を `split(":", 1)` で割って `[1]` を `strip()`。概念2でやった2つの技を順に使うだけ。

In [ ]:
result2 = None
# ここに書く(result2 に代入する。まず中身を切り出し、次に "名前: " を取り除く)


check("概念2: 値の切り出し", result2, "田中 美咲",
      hint='開始タグは \'<h2 class="name">\' の文字列そのまま。切り出した後 split(":", 1)[1].strip() で "名前: " が落ちる')

---
## 概念3: リスト内包表記 — LINQ の Select / Where を1行で

### なぜ学ぶか
解析で切り出した断片は、たいてい**リストのまま一括加工**します:「全リンクのうち内部リンク(`/`始まり)だけ集める」「全タグの前後空白を掃除する」。C# の LINQ を毎日書いている人ほど、この対応を押さえると Python が一気に手に馴染みます。

### 解説

**リスト内包表記**は、LINQ の `Select`(変換)と `Where`(絞り込み)を**1つの式**にまとめた書き方です。これが Python でデータを扱うときの「共通言語」なので、必ず読み書きできるようにします。

```
C#:      xs.Select(x => x.Trim()).ToList()
Python:  [x.strip() for x in xs]
           ^^^^^^^^ 変換部    ^^^^^^^^ ループ部

C#:      xs.Where(x => x.Length >= 5).Select(x => x.ToUpper()).ToList()
Python:  [x.upper() for x in xs if len(x) >= 5]
           ^^^^^^^^ 変換    ^^^^^^^^^^^  ^^^^^^^^^^^^^ 絞り込み(if)
```

読む順番は「**for → if → 先頭の式**」。「`xs` を1つずつ `x` に取り出し(for)、条件に合うものだけ(if)、こう変換する(先頭の式)」。`if` は省略でき、その場合は全要素を変換します(`Select` だけ)。

`len(x)` は文字列やリストの長さ(C# の `.Length` / `.Count`)、`.upper()` は大文字化(C# の `.ToUpper()`)です。

In [ ]:
# GOAL: リスト内包表記で「変換」と「絞り込み」を1行で書けることを確認する

words = ["メニュー", "予約フォーム", "アクセス", "焙煎", "ハンドドリップ"]

# STEP 1: 変換だけ(Select 相当) — 全要素の文字数に変換
lengths = [len(w) for w in words]
print("各語の文字数:", lengths)

# STEP 2: 絞り込みだけ(Where 相当) — 4文字以上の語だけ残す
long_ones = [w for w in words if len(w) >= 4]
print("4文字以上   :", long_ones)

# STEP 3: 変換 + 絞り込み(Where→Select) — 4文字以上を「〈語〉」で囲んで整形
decorated = ["〈" + w + "〉" for w in words if len(w) >= 4]
print("整形結果    :", decorated)

### 予測してみよう

次のセルは、リンクURLのリストから **`/` で始まる内部リンクだけ**を集めます。`w.startswith("/")` は「`/` で始まるか」を返すメソッド(C# の `w.StartsWith("/")`)。

**実行する前に予測**: `links` のうち残るのはどれで、何個でしょう?

In [ ]:
# 予測してから実行!
links = ["/menu", "https://example.com", "/reserve", "mailto:info@x.test", "/access"]
internal = [w for w in links if w.startswith("/")]
print("内部リンクだけ:", internal)
print("個数         :", len(internal))

### 書いてみる

**課題**: 下の `raw_tags`(`split` 直後で各要素に空白が残っている状態)から、**各要素の前後空白を strip したリスト**を内包表記で作り、`result3` に入れてください(期待値: `["焙煎", "ハンドドリップ", "浅煎り"]`)。

ヒント(概念レベル): 絞り込みは不要。`[<各要素を変換する式> for t in raw_tags]` の形。変換は前後空白を取る `t.strip()`。

In [ ]:
raw_tags = [" 焙煎", " ハンドドリップ ", "浅煎り "]

result3 = None
# ここに書く(result3 に代入する。内包表記で各要素を strip)


check("概念3: リスト内包表記", result3, ["焙煎", "ハンドドリップ", "浅煎り"],
      hint="[t.strip() for t in raw_tags] の形。if は不要")

---
## 概念4: 辞書操作 — Dictionary<K,V> で集計する

### なぜ学ぶか
整形([3])の最終形は、たいてい「1レコード = 1つの辞書」です:`{"name": "田中 美咲", "job": "焙煎士"}`。さらに「どのタグが何人に使われているか」のような**集計**も辞書で数えます。CSV出力(unit05)の直前は必ずこの `dict` の形を経由するので、ここは全ユニット共通の合流点です。

### 解説

Python の `dict` は C# の `Dictionary<K, V>` そのものです。

| Python | 何をする | C# の対応 |
|--------|----------|-----------|
| `d = {"name": "田中"}` | 辞書リテラル | `new Dictionary<...>{["name"]="田中"}` |
| `d["name"]` | 値を取り出す(キーが無いとエラー) | `d["name"]` |
| `d.get(k, 既定値)` | 値を取り出す。**キーが無ければ既定値** | `TryGetValue` 相当を1行で |
| `d[k] = v` | 追加・更新 | 同じ |

**集計の定石(出現回数を数える)**は `get` が主役です:

```python
counts = {}
for tag in tags:
    counts[tag] = counts.get(tag, 0) + 1   # 無ければ0から、あれば+1
```

`counts.get(tag, 0)` は「`tag` があればその値、無ければ `0`」を返すので、初回でも安全に `+1` できます。`counts[tag]` と直接書くと初回に**キーが無くて KeyError** になるので、集計では `get` を使うのが鉄則です。

In [ ]:
# GOAL: 辞書での「値の取り出し」と「出現回数の集計」を確認する

# STEP 1: 1レコードを辞書で表す(これが整形のゴールの形)
staff = {"name": "田中 美咲", "job": "焙煎士"}
print("名前:", staff["name"], "/ 職業:", staff["job"])

# STEP 2: get で安全に取り出す。存在しないキー "age" は既定値 "不明" が返る
print("年齢:", staff.get("age", "不明"))   # KeyError にならない

# STEP 3: get を使った出現回数の集計
tags = ["焙煎", "浅煎り", "焙煎", "ハンドドリップ", "浅煎り", "焙煎"]
counts = {}
for tag in tags:
    counts[tag] = counts.get(tag, 0) + 1   # 無ければ0スタート、あれば+1
print("集計結果:", counts)

### 予測してみよう

次のセルは、上の `counts`(`{"焙煎": 3, "浅煎り": 2, "ハンドドリップ": 1}`)に対して `get` を2回呼びます。片方は存在するキー、もう片方は存在しないキーです。

**実行する前に予測**: それぞれ何が表示されるでしょう?

In [ ]:
# 予測してから実行!
print("焙煎の回数    :", counts.get("焙煎", 0))
print("深煎りの回数  :", counts.get("深煎り", 0))   # 存在しないキー

存在しないキーでもエラーにならず既定値 `0` が返る — これが `get` を使う理由です。

### 書いてみる

**課題**: 下の `tag_lists`(スタッフ2人分のタグリスト)を走査し、**タグごとに何人が使っているか**を数えた辞書を `result4` に入れてください(期待値: `{"焙煎": 2, "浅煎り": 1, "深煎り": 1}`)。

ヒント(概念レベル): 空の辞書 `{}` を用意し、二重ループ(`for tags in tag_lists:` の中で `for tag in tags:`)で `counts[tag] = counts.get(tag, 0) + 1` を回す。解説の集計の定石を、リストのリストに広げるだけ。

In [ ]:
tag_lists = [["焙煎", "浅煎り"], ["焙煎", "深煎り"]]

result4 = None
# ここに書く(result4 に代入する。get を使ってタグごとの人数を集計)


check("概念4: 辞書で集計", result4, {"焙煎": 2, "浅煎り": 1, "深煎り": 1},
      hint="外側 for tags in tag_lists、内側 for tag in tags。counts[tag] = counts.get(tag, 0) + 1")

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | C#で言うと |
|------|--------|-----------|
| 全体像 / DOMツリー | スクレイピングは 取得→解析→整形→出力。HTMLは入れ子の箱 | オブジェクトグラフ |
| 文字列操作 | `find`+スライス`s[a:b]`で切り出し、`strip`/`split`で掃除・分割 | `IndexOf`/`Substring`/`Trim`/`Split` |
| リスト内包表記 | `[式 for x in xs if 条件]` で変換+絞り込みを1行 | LINQ `Select`/`Where` |
| 辞書操作 | `d.get(k, 既定値)` で安全に取り出し・集計 | `Dictionary` / `TryGetValue` |

**この先どこで使うか**:
- **unit02** で BeautifulSoup を使うと、今日の `find`+スライスでの「タグの中身の切り出し」が `soup.find("h1").text` の一撃に置き換わります。今日わざと素朴にやったことで、**この文字列処理がいかに壊れやすいか**(タグの書き方が少し変わると即バグ)を体感したはず。その壊れやすさをライブラリが解決してくれる、というのが次の話です。
- リスト内包表記と辞書は unit02 以降**毎回**登場します(複数レコードを `list[dict]` に整形 → unit05 でCSVに書き出す)。今日の集計が実データ整形の土台になります。

**次**: 演習 `ex01_string_extract.py` へ。lesson を見ながらで OK。テストは
`python -m pytest courses/web-scraping/unit01-html-and-python-basics/tests/test_ex01.py -q`